# Document-Type Classification (eDiscovery Triage) — Training Notebook

This notebook fine-tunes a small model (`distilbert-base-uncased`) to sort documents into
3 types:

- **Contract**
- **Email**
- **Other**

**Why this task:** before anyone can review a bulk document dump in an eDiscovery matter,
it first needs to be sorted by type — that triage step is what this model does.

**Important caveat about the "Other" class:** there is no clean public dataset of generic
business memos, letters, or invoices, so "Other" is trained on ordinary news-article text
instead, purely as a stand-in. This teaches the model "this is neither a contract nor an
email" — it does **not** mean the model has learned to recognize memos, invoices, or any
other real document type. Treat "Other" as a placeholder category, not a validated
eDiscovery classification.

**How it works (simple version):** unlike the clause-extraction notebook (which searches
*inside* a contract for a specific span of text), this is a more standard classification
task — the model reads a document and picks one of the 3 labels above, the same way a spam
filter picks "spam" or "not spam."

**How to use this notebook:**
1. Open this file in Google Colab (colab.research.google.com -> File -> Upload notebook).
2. Go to `Runtime > Change runtime type` and set **Hardware accelerator** to **GPU** (the free T4 is fine).
3. Go to `Runtime > Run all`.
4. Wait for training to finish — the cells run top to bottom automatically, no input needed.
5. At the very end, your browser will download a file called `document_classification_model.zip` —
   that is your trained model.

You do not need to upload any files yourself — all 3 training datasets download
automatically inside this notebook.


In [ ]:
!pip install -q transformers datasets accelerate evaluate

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU is ON: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected!")
    print("Go to: Runtime > Change runtime type > Hardware accelerator > GPU, then run this cell again.")


## Step 1: Build the training dataset

There is no single ready-made dataset labeled "Contract vs Email vs Other," so this
notebook assembles one from 3 unrelated public datasets — using only the raw text from
each, and throwing away whatever label that dataset originally had. Each source becomes
one of our 3 classes.


In [ ]:
# --- Contract: CUAD (same dataset used in notebook 02) ---
# CUAD's annotation file repeats each contract's full text once per question asked about
# it, so we dedupe by contract title to get one text per unique contract.
from huggingface_hub import hf_hub_download
import json

cuad_json_path = hf_hub_download(
    repo_id="theatticusproject/cuad",
    repo_type="dataset",
    filename="CUAD_v1/CUAD_v1.json",
)

with open(cuad_json_path, encoding="utf-8") as f:
    cuad_raw = json.load(f)

seen_titles = set()
contract_texts = []
for document in cuad_raw["data"]:
    if document["title"] in seen_titles:
        continue
    seen_titles.add(document["title"])
    full_text = "\n".join(p["context"] for p in document["paragraphs"])
    if full_text.strip():
        contract_texts.append(full_text)

print(f"Unique contracts: {len(contract_texts)}")


In [ ]:
# --- Email: Enron corporate emails ---
# We only use the email body text here - the spam/ham label this dataset ships with is
# irrelevant to us, since every row is a real email regardless of that label.
from datasets import load_dataset

enron = load_dataset("SetFit/enron_spam", split="train")
email_texts = [t for t in enron["text"] if t and t.strip()]

print(f"Emails available: {len(email_texts)}")


In [ ]:
# --- Other: news articles (placeholder/proxy class, see the caveat above) ---
# Again, we ignore the dataset's own label (news category) - we only want generic prose
# that is clearly not a contract and not an email, to give the model a contrast class.
# Note: use the namespaced repo id ("fancyzhx/ag_news"), not the old bare "ag_news" -
# recent huggingface_hub versions reject un-namespaced dataset ids.
ag_news = load_dataset("fancyzhx/ag_news", split="train")
other_texts = [t for t in ag_news["text"] if t and t.strip()]

print(f"'Other' examples available: {len(other_texts)}")


In [ ]:
# --- Balance the 3 classes and combine into one dataset ---
import random

random.seed(42)

# CUAD only has ~500 unique contracts total, so that's the natural ceiling - capping every
# class at the same number keeps the 3 classes balanced and keeps training fast on Colab's
# free GPU (a few hundred examples per class trains in minutes, not hours).
PER_CLASS_CAP = 500


def sample(items, cap):
    items = list(items)
    random.shuffle(items)
    return items[:cap]


contract_sample = sample(contract_texts, PER_CLASS_CAP)
email_sample = sample(email_texts, PER_CLASS_CAP)
other_sample = sample(other_texts, PER_CLASS_CAP)

from datasets import Dataset

LABELS = ["Contract", "Email", "Other"]
id2label = {i: label for i, label in enumerate(LABELS)}
label2id = {label: i for i, label in enumerate(LABELS)}

records = (
    [{"text": t, "label": label2id["Contract"]} for t in contract_sample]
    + [{"text": t, "label": label2id["Email"]} for t in email_sample]
    + [{"text": t, "label": label2id["Other"]} for t in other_sample]
)

full_dataset = Dataset.from_list(records).shuffle(seed=42)
split_dataset = full_dataset.train_test_split(test_size=0.15, seed=42)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

print(f"Train examples: {len(train_dataset)}")
print(f"Validation examples: {len(val_dataset)}")


## Step 2: Turn text into numbers (tokenization)

Same idea as notebook 02, but simpler: since we just need the document's overall type
(not a specific span of text inside it), we don't need the sliding-window trick from the
clause-extraction notebook. The document type is usually obvious from the first page or
two, so we just take the beginning of the text and cut it off at a fixed length.


In [ ]:
from transformers import AutoTokenizer

MODEL_CHECKPOINT = "distilbert-base-uncased"
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)


def preprocess(examples):
    return tokenizer(examples["text"], max_length=MAX_LENGTH, truncation=True, padding="max_length")


train_tokenized = train_dataset.map(preprocess, batched=True, remove_columns=["text"])
val_tokenized = val_dataset.map(preprocess, batched=True, remove_columns=["text"])

print(f"Tokenized train examples: {len(train_tokenized)}")
print(f"Tokenized validation examples: {len(val_tokenized)}")


## Step 3: Fine-tune the model

Same training setup as notebook 02 (same base model, same learning rate, same number of
epochs) — this is an easier task than clause extraction, so there's no need to change
those settings. The one difference: because this task has a simple right/wrong answer per
document, we can track real accuracy and F1 scores during training, not just loss.


In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
import numpy as np
import evaluate

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=3, id2label=id2label, label2id=label2id
)

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        **accuracy_metric.compute(predictions=predictions, references=labels),
        **f1_metric.compute(predictions=predictions, references=labels, average="macro"),
    }


training_args = TrainingArguments(
    output_dir="document-classification-checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
    # Newer transformers versions renamed this argument from `tokenizer` to
    # `processing_class` (passing `tokenizer=` now raises a TypeError).
    processing_class=tokenizer,
)

trainer.train()


## Step 4: Sanity check — does it actually work?

Same principle as notebook 02: never trust model output blindly. Let's manually compare a
few predictions against the real label before moving on.


In [ ]:
# We decode the prediction ourselves instead of using transformers' pipeline("text-classification", ...)
# helper, to keep this notebook consistent with notebook 02's approach and avoid relying on
# a pipeline API that can change between transformers versions.
device = next(model.parameters()).device


def predict_label(text: str):
    inputs = tokenizer(text, max_length=MAX_LENGTH, truncation=True, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=-1)[0]
    predicted_id = int(torch.argmax(probs).item())
    predicted_label = model.config.id2label[predicted_id]
    all_probs = {model.config.id2label[i]: round(probs[i].item(), 2) for i in range(len(probs))}
    return predicted_label, all_probs


for i in range(5):
    example = val_dataset[i]
    predicted_label, probs = predict_label(example["text"])
    print("=" * 80)
    print("Text snippet:", example["text"][:200].replace("\n", " "), "...")
    print("Predicted:", predicted_label, probs)
    print("Actual:   ", id2label[example["label"]])


## Step 5: Save and download your trained model

Same as notebook 02 — this packages the trained model into a single zip file and
downloads it straight to your computer.


In [ ]:
import shutil

SAVE_DIR = "document-classification-model-final"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

shutil.make_archive("document_classification_model", "zip", SAVE_DIR)
print("Saved and zipped to document_classification_model.zip")

from google.colab import files
files.download("document_classification_model.zip")
